In [1]:
import sys
import subprocess
from pathlib import Path

REPO = Path("ProPainter")

# Clonar ProPainter.
if not REPO.exists():
    subprocess.check_call([
        "git",
        "clone",
        "https://github.com/sczhou/ProPainter.git",
        str(REPO),
    ])

# Actualizar pip en el mismo Python del notebook.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel",
])

# PyTorch con CUDA 12.6 para la RTX 4090.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "torch==2.6.0",
    "torchvision==0.21.0",
    "--index-url",
    "https://download.pytorch.org/whl/cu126",
])

# Dependencias de ProPainter.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-r",
    str(REPO / "requirements.txt"),
])

# Confirmar que PyTorch ve la GPU.
import torch

print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch no detectó la GPU. Reinicia el kernel y ejecuta nuevamente."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
    "GB",
)

PyTorch: 2.6.0+cu126
CUDA de PyTorch: 12.6
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 23.99 GB


In [ ]:
import os
import sys
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [
        sys.executable,
        "-u",
        "inference_propainter.py",
        "--video", "../Videos/video30min-11to22/frames",
        "--mask", "../Videos/video30min-11to22/masks",
        "--output", "../Videos/video30min-11to22/no_hud",
        "--width", "1280",
        "--height", "720",
        "--fp16",
        "--save_frames",
        "--neighbor_length", "10",
        "--ref_stride", "10",
        "--subvideo_length", "40",
        "--mask_dilation", "3",
        "--raft_iter", "20",
    ],
    cwd="ProPainter",
    check=True,
)